# S14 — Validación externa en CICIoT2023 (punto 3 de la revisión del docente)
Aplica **el mismo procedimiento** del artículo a un segundo conjunto independiente, CICIoT2023 (Neto et al., *Sensors* 2023):
auditoría de entropía (información mutua frente a la entropía de la etiqueta, pureza de una columna y conflictos de etiqueta),
ranking de IM con la regla de parsimonia, balanceo elegido por CV, MI-LiteRF y MI-LiteXGB **con la configuración del artículo, sin reajustar**,
y la TCN de Gao et al. con su configuración original sobre las mismas entradas. Todo se guarda en Drive, en `mejoras/S14_CICIoT2023/`.

| Parte | Qué hace | Dónde | Tiempo aprox. |
|---|---|---|---|
| **A0** | Descarga (Kaggle) o lee de Drive los CSV originales, toma una muestra de archivos, limpia y guarda `cic_muestra.parquet` en Drive | CPU, RAM alta | 10–20 min (solo la 1.ª vez) |
| **A1** | Auditoría: IM y pureza de cada atributo, conflictos de etiqueta y cota del oráculo | CPU | ≈ 3 min |
| **A2** | 10 repeticiones 70/30: ranking de IM, *k* (CV en la REP-01), balanceo por CV, MI-LiteRF y MI-LiteXGB | CPU | ≈ 5–8 min por REP |
| **A3** | TCN de Gao (sin cambios) con las mismas entradas de IM | **GPU** | ≈ 8–12 min por REP |
| **A4** | Ablación: XGBoost sin los marcadores que señale A1 y sin el atributo de mayor IM | CPU | ≈ 2 min por REP |
| **A5** | Latencia por paquete (lote de 1, ONNX Runtime, 1 hilo) con los modelos de la REP-01 | CPU | ≈ 3 min |
| **A6** | Resumen, pruebas pareadas (t remuestreada corregida + Holm) y `S14_CICIoT2023_resultados.json` | Cualquiera | < 1 min |

**Cómo correrlo (lo más simple):** una sola sesión **GPU con RAM alta** (L4 o A100), *Entorno de ejecución → Ejecutar todo*.
Las partes de CPU también corren ahí. Si la sesión se corta, vuelve a ejecutar todo: con `REANUDAR = True` retoma donde quedó.
Para repartir: `REPS = [1, 2, 3, 4, 5]` en una sesión y `[6, 7, 8, 9, 10]` en otra (la REP-01 debe terminar A2 primero, porque fija *k*).

**Datos.** Por defecto se descargan de Kaggle con `kagglehub` (espejo de los CSV originales de la UNB). Si Kaggle pide credenciales,
agrega `KAGGLE_USERNAME` y `KAGGLE_KEY` en *Secretos* de Colab (ícono de llave), o copia a mano algunos `part-*.csv` a la carpeta
`Articulo_Grupal_EdgeIIoT/CICIoT2023_csv/` de Drive: el cuaderno usa esa carpeta si encuentra CSV en ella.

Al final, **comparte el archivo `S14_CICIoT2023_resultados.json`** (o la salida de A6) para integrar los resultados al artículo.

In [ ]:
import os
# --- Qué se ejecuta ---------------------------------------------------------------------------------
EJECUTAR = {'A1': True, 'A2': True, 'A3': True, 'A4': True, 'A5': True, 'A6': True}   # A3 necesita GPU
REPS = list(range(1, 11))
REANUDAR = True              # reutiliza lo ya guardado en Drive (por parte y por REP)
MODO_PRUEBA = os.environ.get('S14_PRUEBA') == '1'   # datos sintéticos pequeños, sin Drive ni Kaggle (solo para probar el cuaderno)

# --- Rutas ------------------------------------------------------------------------------------------
CARPETA = os.environ.get('CARPETA_ARTICULO', '/content/drive/MyDrive/Articulo_Grupal_EdgeIIoT')
DIR_CSV_DRIVE = os.path.join(CARPETA, 'CICIoT2023_csv')     # opcional: CSV originales copiados a mano
KAGGLE_SLUG = 'madhavmalhotra/unb-cic-iot-dataset'           # espejo en Kaggle de los CSV originales (part-*.csv)
DIR_X = os.path.join(CARPETA, 'mejoras', 'S14_CICIoT2023')    # salida
DIR_TMP = os.environ.get('DIR_TMP_S14', '/content/tmp_s14')

# --- Muestra y tarea (fijadas antes de mirar resultados) --------------------------------------------
N_ARCHIVOS = 5               # CSV usados de los ~169 (a intervalos regulares de la lista ordenada)
TAREA = '8clases'            # '8clases' (Benign + 7 categorías de la UNB) o '34clases' (cada ataque)
SEMILLA_BASE = 1000          # REP r usa la semilla 1000 + r, como en el artículo
TEST_SIZE = 0.30
# --- Igual que el artículo ---------------------------------------------------------------------------
MI_SUBMUESTRA = 200_000      # paquetes de entrenamiento para estimar la IM
MI_MAX_DISCRETO = 64         # atributos con ≤ 64 valores se usan tal cual; el resto, 32 bins por cuantiles
MI_BINS = 32
K_GRID = [5, 10, 15, 20, 25, 30]
T_GRID = [2000, 20000, None] # None = sin balanceo (distribución natural)
TOLERANCIA = 0.99            # regla de parsimonia: CV ≥ 99 % del mejor y, entre esos, el modelo más pequeño
CV_SUBMUESTRA = 400_000      # filas de entrenamiento (estratificadas) para la CV de k y de T (por costo)
SMOTE_K = 5
RF_CFG = dict(arboles=25, profundidad=15)                    # MI-LiteRF del artículo
XGB_CFG = dict(arboles=50, profundidad=4, lr=0.1)           # MI-LiteXGB del artículo
TCN_CFG = dict(lr=1e-3, epocas=50, lote=128, dilataciones=[1, 2, 4, 8], kernel=1, filtros=26, dropout=0.1, val_split=0.1)
# --- Auditoría (A1) ------------------------------------------------------------------------------------
UMBRAL_PUREZA = 0.999        # marcador candidato si una sola columna acierta Normal/Ataque en ≥ 99,9 % de las filas
UMBRAL_IM_REL = 0.90         # ... o si su IM con Normal/Ataque alcanza el 90 % de H(Y_b)
# --- Latencia (A5) --------------------------------------------------------------------------------------
N_LOTE1 = 10_000
CALENTAMIENTO = 500
ALFA = 0.05

if MODO_PRUEBA:   # tamaños mínimos para probar el flujo completo en minutos
    CARPETA = os.environ.get('CARPETA_PRUEBA', '/tmp/prueba_s14')
    DIR_X = os.path.join(CARPETA, 'mejoras', 'S14_CICIoT2023'); DIR_CSV_DRIVE = os.path.join(CARPETA, 'CICIoT2023_csv')
    DIR_TMP = os.path.join(CARPETA, 'tmp'); REPS = [1, 2]; MI_SUBMUESTRA = 20_000; CV_SUBMUESTRA = 20_000
    K_GRID = [5, 10, 15]; TCN_CFG = dict(TCN_CFG, epocas=2, lote=256); N_LOTE1 = 300; CALENTAMIENTO = 20
os.makedirs(DIR_TMP, exist_ok=True)
print('REPS:', REPS, '| partes:', [p for p, v in EJECUTAR.items() if v], '| tarea:', TAREA, '| salida:', DIR_X)

## 0. Drive, versiones y librerías
Se fijan scikit-learn, imbalanced-learn y XGBoost en las versiones del artículo. Si Colab trae otras, la celda las instala
y pide reiniciar la sesión una vez.

In [ ]:
import sys, subprocess, json, time, platform, datetime, io, gc, glob, gzip, shutil, warnings, math
import importlib.metadata as im
from types import SimpleNamespace
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB and not MODO_PRUEBA and not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not MODO_PRUEBA and not os.path.isdir(CARPETA):
    raise SystemExit(f'⛔ No encuentro {CARPETA}. Revisa que el Drive montado sea el de la cuenta dueña de Articulo_Grupal_EdgeIIoT.')
os.makedirs(DIR_X, exist_ok=True)

def version(paquete):
    try:
        return im.version(paquete)
    except im.PackageNotFoundError:
        return None

V_ARTICULO = {'scikit-learn': '1.6.1', 'imbalanced-learn': '0.14.2', 'xgboost': '3.4.1'}
if EN_COLAB and not MODO_PRUEBA:
    distintas = {p: version(p) for p, v in V_ARTICULO.items() if version(p) != v}
    if distintas:
        ya = [m for m in ('sklearn', 'imblearn', 'xgboost') if m in sys.modules]
        print('Instalando las versiones del artículo (ahora hay', distintas, ')')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *[f'{p}=={v}' for p, v in V_ARTICULO.items()]], check=True)
        if ya:
            raise SystemExit('⛔ Se cambiaron librerías ya cargadas: Entorno de ejecución → Reiniciar sesión, y vuelve a ejecutar todo.')
    faltan = [p for p in ('onnx', 'onnxruntime', 'skl2onnx', 'onnxmltools', 'kagglehub', 'pyarrow') if version(p) is None]
    if faltan:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltan], check=True)
    if version('tf2onnx') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'tf2onnx'], check=True)

import numpy as np, pandas as pd, joblib
import sklearn, imblearn, xgboost, scipy
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy import stats
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

def cpu_modelo():
    try:
        for l in open('/proc/cpuinfo'):
            if l.startswith('model name'):
                return l.split(':', 1)[1].strip()
    except Exception:
        pass
    return platform.processor()

def hay_gpu():
    return bool(shutil.which('nvidia-smi')) and 'GPU' in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout

ENTORNO = {'fecha_hora': datetime.datetime.now().strftime('%d/%m/%Y %H:%M'), 'procesador': cpu_modelo(),
           'nucleos_logicos': os.cpu_count(), 'gpu': hay_gpu(), 'python': platform.python_version(),
           'numpy': np.__version__, 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
           'imbalanced_learn': imblearn.__version__, 'xgboost': xgboost.__version__, 'scipy': scipy.__version__,
           'tensorflow': version('tensorflow'), 'onnxruntime': version('onnxruntime'), 'modo_prueba': MODO_PRUEBA}
print(json.dumps(ENTORNO, indent=1))

## 1. Utilidades (las mismas del artículo)

In [ ]:
def guardar_json(obj, ruta):
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1,
                  default=lambda o: o.item() if hasattr(o, 'item') else (o.tolist() if hasattr(o, 'tolist') else str(o)))

def leer_json(ruta):
    with open(ruta, encoding='utf-8') as f:
        return json.load(f)

def hay(ruta):
    return REANUDAR and os.path.exists(ruta)

def correr(parte):
    if not EJECUTAR.get(parte):
        print(f'⏭️ {parte} desactivada en EJECUTAR.')
        return False
    return True

def entropia(*codigos):
    c = codigos[0].astype(np.int64)
    for otro in codigos[1:]:
        c = c * (int(otro.max()) + 1) + otro.astype(np.int64)
    _, cnt = np.unique(c, return_counts=True)
    p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())

def discretizar(x):
    u = np.unique(x)
    if len(u) <= MI_MAX_DISCRETO:
        return np.searchsorted(u, x)
    bordes = np.unique(np.quantile(x, np.linspace(0, 1, MI_BINS + 1)[1:-1]))
    return np.digitize(x, bordes)

def info_mutua(xd, y):
    return entropia(y) + entropia(xd) - entropia(xd, y)

def pureza(xd, yb):
    """Exactitud de predecir Normal/Ataque con la etiqueta mayoritaria de cada valor (prueba de una columna de Galala)."""
    tab = pd.crosstab(xd, yb)
    return float(tab.max(axis=1).sum() / tab.values.sum())

def balancear(Xs, ys, T, semilla):
    cnt = np.bincount(ys, minlength=N_CLASES)
    bajo = {c: T for c in range(N_CLASES) if cnt[c] > T}
    if bajo:
        Xs, ys = RandomUnderSampler(sampling_strategy=bajo, random_state=semilla).fit_resample(Xs, ys)
    cnt = np.bincount(ys, minlength=N_CLASES)
    sobre = {c: T for c in range(N_CLASES) if 0 < cnt[c] < T}
    if sobre:
        kn = max(1, min(SMOTE_K, int(cnt[list(sobre)].min()) - 1))
        Xs, ys = SMOTE(sampling_strategy=sobre, k_neighbors=kn, random_state=semilla).fit_resample(Xs, ys)
    Xs, ys = TomekLinks(sampling_strategy='all', n_jobs=-1).fit_resample(Xs, ys)
    return np.ascontiguousarray(Xs, dtype=np.float32), ys

def datos_T(Xs, ys, T, semilla):
    if not T:
        return np.ascontiguousarray(Xs, dtype=np.float32), ys
    return balancear(Xs, ys, int(T), semilla)

def T_txt(T):
    return 'sin balanceo' if not T else f'{int(T):,}'.replace(',', ' ')

def indicadores(y_real, y_pred):
    cm = confusion_matrix(y_real, y_pred, labels=range(N_CLASES))
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0) * 100
    TN = cm[NORMAL, NORMAL]; FP = cm[NORMAL].sum() - TN
    ataques = [c for c in range(N_CLASES) if c != NORMAL]
    miss = cm[ataques, NORMAL].sum() / max(cm[ataques].sum(), 1) * 100
    en = []
    for c in ataques:
        if cm[c].sum() > 0:
            m = cm[c, NORMAL] / cm[c].sum()
            if m < 1:
                en.append(m / (1 - m))
    return {'f1_macro': float(f1.mean()), 'f1_por_clase': dict(zip(CLASES, f1.tolist())),
            'tfp': float(FP / max(FP + TN, 1) * 100), 'falsas_alarmas': int(FP), 'ataques_como_normal_pct': float(miss),
            'E_N_registros': float(np.mean(en)) if en else None,
            'exactitud': float(accuracy_score(y_real, y_pred) * 100),
            'f1_ponderado': float(f1_score(y_real, y_pred, average='weighted') * 100),
            'matriz_confusion': cm.tolist()}

def f1_macro(y_real, y_pred):
    return f1_score(y_real, y_pred, average='macro', labels=range(N_CLASES), zero_division=0) * 100

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p)
    orden = np.argsort(p, kind='mergesort'); ajust = np.empty(m); acumulado = 0.0
    for i, idx in enumerate(orden):
        acumulado = max(acumulado, min(1.0, (m - i) * p[idx])); ajust[idx] = acumulado
    return ajust

def t_corregida(d, n_tr, n_te):
    """t remuestreada corregida (Nadeau y Bengio, 2003)."""
    d = np.asarray(d, float); J = len(d); m = float(d.mean())
    s2 = float(d.var(ddof=1)) if J > 1 else 0.0
    se = np.sqrt((1 / J + n_te / n_tr) * s2)
    t = m / se if se > 0 else np.inf
    h = stats.t.ppf(0.975, max(J - 1, 1)) * se
    return {'media_d': m, 't_corr': float(t), 'gl': J - 1, 'p_corr': float(2 * stats.t.sf(abs(t), max(J - 1, 1))),
            'ic95_corr': [round(float(m - h), 4), round(float(m + h), 4)]}

def mostrar(df):
    try:
        display(df)
    except NameError:
        print(df.to_string())

def ms(x):
    x = np.asarray(x, float)
    return [float(x.mean()), float(x.std(ddof=1)) if len(x) > 1 else 0.0]

def kb_joblib(modelo):
    b = io.BytesIO(); joblib.dump(modelo, b); return b.tell() / 1024

def nuevo_rf(semilla):
    return RandomForestClassifier(RF_CFG['arboles'], max_depth=RF_CFG['profundidad'], n_jobs=-1, random_state=semilla)

def nuevo_xgb(semilla):
    return XGBClassifier(n_estimators=XGB_CFG['arboles'], max_depth=XGB_CFG['profundidad'], learning_rate=XGB_CFG['lr'],
                         tree_method='hist', device='cpu', n_jobs=-1, random_state=semilla, eval_metric='mlogloss')
FABRICA = {'rf': nuevo_rf, 'xgb': nuevo_xgb}

## A0. Datos: descarga, muestra de archivos y limpieza
Se usan `N_ARCHIVOS` CSV originales elegidos a intervalos regulares de la lista ordenada (cada archivo mezcla todas las clases).
Limpieza igual que en Edge-IIoTset: se quitan filas con valores faltantes o no finitos y luego los duplicados.
Si el CSV trae una marca de tiempo (`ts`), se retira, como `frame.time` en Edge-IIoTset.

In [ ]:
CAT_34_A_8 = {
    'benigntraffic': 'Benign',
    **{k: 'DDoS' for k in ['ddos-rstfinflood', 'ddos-pshack_flood', 'ddos-syn_flood', 'ddos-udp_flood', 'ddos-tcp_flood',
                            'ddos-icmp_flood', 'ddos-synonymousip_flood', 'ddos-ack_fragmentation', 'ddos-udp_fragmentation',
                            'ddos-icmp_fragmentation', 'ddos-slowloris', 'ddos-http_flood']},
    **{k: 'DoS' for k in ['dos-udp_flood', 'dos-syn_flood', 'dos-tcp_flood', 'dos-http_flood']},
    **{k: 'Mirai' for k in ['mirai-greeth_flood', 'mirai-greip_flood', 'mirai-udpplain']},
    **{k: 'Recon' for k in ['recon-pingsweep', 'recon-osscan', 'recon-portscan', 'vulnerabilityscan', 'recon-hostdiscovery']},
    **{k: 'Spoofing' for k in ['dns_spoofing', 'mitm-arpspoofing']},
    **{k: 'Web' for k in ['browserhijacking', 'backdoor_malware', 'xss', 'uploading_attack', 'sqlinjection', 'commandinjection']},
    'dictionarybruteforce': 'BruteForce'}

def categoria(etiqueta):
    e = str(etiqueta).strip().lower()
    if e in CAT_34_A_8:
        return CAT_34_A_8[e]
    for pref, cat in (('ddos', 'DDoS'), ('dos', 'DoS'), ('mirai', 'Mirai'), ('recon', 'Recon'), ('benign', 'Benign')):
        if e.startswith(pref):
            return cat
    return None

def datos_sinteticos(ruta, n=60_000, semilla=7):
    """Solo para MODO_PRUEBA: CSV con el formato de CICIoT2023 (46 atributos + label) y 34 etiquetas."""
    rng = np.random.default_rng(semilla)
    cols = ['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 'fin_flag_number',
            'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number',
            'ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC',
            'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number',
            'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight']
    etiquetas = list(CAT_34_A_8.keys())
    p = np.array([0.03] + [0.9 / 33] * 33); p = p / p.sum()
    nombres = {k: k for k in etiquetas}
    nombres.update({'benigntraffic': 'BenignTraffic', 'ddos-rstfinflood': 'DDoS-RSTFINFlood', 'xss': 'XSS'})
    y = rng.choice(len(etiquetas), size=n, p=p)
    X = rng.normal(size=(n, len(cols))) + (y[:, None] % 7) * 0.4 * (np.arange(len(cols)) % 5 == 0)
    X[:, cols.index('IAT')] = 8.3e7 + y * 1e4 + rng.normal(scale=50, size=n)   # marcador de captura, como sospechamos
    X[:, cols.index('Protocol Type')] = rng.integers(0, 17, size=n)
    df = pd.DataFrame(X.astype(np.float32), columns=cols)
    df['label'] = [nombres[etiquetas[i]] for i in y]
    df.iloc[:50, 4] = np.inf                                                           # filas no finitas
    df = pd.concat([df, df.iloc[:200]], ignore_index=True)                            # duplicados
    os.makedirs(ruta, exist_ok=True)
    df = df.sample(frac=1, random_state=1).reset_index(drop=True)
    for i, idx in enumerate(np.array_split(np.arange(len(df)), 8)):
        df.iloc[idx].to_csv(os.path.join(ruta, f'part-{i:05d}-prueba.csv'), index=False)

def localizar_csv():
    archivos = sorted(glob.glob(os.path.join(DIR_CSV_DRIVE, '**', '*.csv'), recursive=True))
    if archivos:
        return archivos, f'Drive ({DIR_CSV_DRIVE})'
    if MODO_PRUEBA:
        datos_sinteticos(DIR_CSV_DRIVE)
        return sorted(glob.glob(os.path.join(DIR_CSV_DRIVE, '*.csv'))), 'sintéticos (MODO_PRUEBA)'
    try:
        import kagglehub
        print(f'Descargando {KAGGLE_SLUG} con kagglehub (≈ 14 GB; solo la primera vez)…')
        ruta = kagglehub.dataset_download(KAGGLE_SLUG)
    except Exception as e:
        raise SystemExit(f'⛔ No se pudo descargar de Kaggle ({type(e).__name__}: {str(e)[:200]}).\n'
                         '   Opción 1: en Colab, ícono de llave (Secretos) → agrega KAGGLE_USERNAME y KAGGLE_KEY y vuelve a ejecutar.\n'
                         f'   Opción 2: copia a mano algunos part-*.csv de CICIoT2023 a {DIR_CSV_DRIVE} y vuelve a ejecutar.')
    return sorted(glob.glob(os.path.join(ruta, '**', '*.csv'), recursive=True)), f'Kaggle ({KAGGLE_SLUG})'

def columna_etiqueta(cols):
    for c in cols:
        if c.strip().lower() == 'label':
            return c
    return None

RUTA_MUESTRA = os.path.join(DIR_X, 'cic_muestra.parquet')
RUTA_A0 = os.path.join(DIR_X, 'A0_datos.json')
if hay(RUTA_MUESTRA) and hay(RUTA_A0):
    A0 = leer_json(RUTA_A0)
    print('✔ Muestra ya preparada:', A0['archivos_usados'], 'archivos ·', f"{A0['filas_depuradas']:,} filas")
else:
    t0 = time.time()
    todos, origen = localizar_csv()
    validos = []
    for a in todos:
        try:
            c = pd.read_csv(a, nrows=0).columns
        except Exception:
            continue
        if columna_etiqueta(c) is not None and len(c) >= 40:
            validos.append(a)
    if not validos:
        raise SystemExit(f'⛔ No encontré CSV de CICIoT2023 (con columna «label» y ≥ 40 columnas) en {origen}.')
    elegidos = sorted({validos[i] for i in np.linspace(0, len(validos) - 1, min(N_ARCHIVOS, len(validos))).round().astype(int)})
    print(f'{origen}: {len(validos)} CSV válidos; se usan {len(elegidos)}:', [os.path.basename(a) for a in elegidos])
    partes = []
    for a in elegidos:
        d = pd.read_csv(a, low_memory=False)
        d.columns = [c.strip() for c in d.columns]
        partes.append(d)
    df = pd.concat(partes, ignore_index=True); del partes
    col_y = columna_etiqueta(df.columns)
    n_bruto = len(df)
    retiradas = [c for c in df.columns if c.lower() in ('ts', 'timestamp')]
    df = df.drop(columns=retiradas)
    atributos = [c for c in df.columns if c != col_y]
    for c in atributos:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df[atributos] = df[atributos].astype(np.float32)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(axis=0, how='any')
    n_finitas = len(df)
    df = df.drop_duplicates(keep='first')
    n_dep = len(df)
    df['etiqueta34'] = df[col_y].astype(str).str.strip()
    df['categoria'] = df['etiqueta34'].map(categoria)
    sin_cat = sorted(df.loc[df['categoria'].isna(), 'etiqueta34'].unique().tolist())
    if sin_cat:
        print('⚠️ Etiquetas sin categoría (se excluyen):', sin_cat)
        df = df[df['categoria'].notna()]
    df = df.drop(columns=[col_y]).reset_index(drop=True)
    df.to_parquet(RUTA_MUESTRA, index=False)
    A0 = {'origen': origen, 'archivos_validos': len(validos), 'archivos_usados': len(elegidos),
          'archivos': [os.path.basename(a) for a in elegidos], 'filas_brutas': int(n_bruto), 'filas_finitas': int(n_finitas),
          'filas_depuradas': int(len(df)), 'duplicados_eliminados': int(n_finitas - n_dep), 'columnas_retiradas': retiradas,
          'atributos': atributos, 'etiquetas_sin_categoria': sin_cat,
          'clases_34': df['etiqueta34'].value_counts().to_dict(), 'clases_8': df['categoria'].value_counts().to_dict(),
          'minutos': round((time.time() - t0) / 60, 1)}
    guardar_json(A0, RUTA_A0)
    del df; gc.collect()
    print(f"✔ {A0['filas_depuradas']:,} filas depuradas de {A0['filas_brutas']:,} ({A0['duplicados_eliminados']:,} duplicados) "
          f"en {A0['minutos']} min")

D = pd.read_parquet(RUTA_MUESTRA)
ATRIBUTOS = [c for c in D.columns if c not in ('etiqueta34', 'categoria')]
col_clase = 'categoria' if TAREA == '8clases' else 'etiqueta34'
CLASES = sorted(D[col_clase].unique().tolist())
CLASES = ['Benign'] + [c for c in CLASES if c != 'Benign'] if 'Benign' in CLASES else \
         (['BenignTraffic'] + [c for c in CLASES if c != 'BenignTraffic'])
N_CLASES = len(CLASES)
NORMAL = 0
X_TODO = D[ATRIBUTOS].to_numpy(np.float32)
Y_TODO = D[col_clase].map({c: i for i, c in enumerate(CLASES)}).to_numpy(np.int64)
YB_TODO = (Y_TODO != NORMAL).astype(np.int64)
del D; gc.collect()
print(f'{len(Y_TODO):,} filas · {len(ATRIBUTOS)} atributos · {N_CLASES} clases')
print(pd.Series(np.bincount(Y_TODO, minlength=N_CLASES), index=CLASES).to_string())

## A1. Auditoría de entropía (sin entrenar)
Para cada atributo: IM con Normal/Ataque y con la clase, comparada con la entropía de la etiqueta, y pureza de una columna.
Un atributo es **marcador candidato** si su pureza es ≥ 0,999 o su IM alcanza el 90 % de H(Y_b) (reglas fijadas antes).
Conflictos de etiqueta: filas con el mismo vector de atributos y etiquetas distintas; el oráculo de mayoría da una cota (en la muestra).

In [ ]:
RUTA_A1 = os.path.join(DIR_X, 'A1_auditoria.json')
if correr('A1'):
    if hay(RUTA_A1):
        A1 = leer_json(RUTA_A1)
    else:
        t0 = time.time()
        H_yb, H_y = entropia(YB_TODO), entropia(Y_TODO)
        filas = []
        for j, a in enumerate(ATRIBUTOS):
            xd = discretizar(X_TODO[:, j])
            filas.append({'atributo': a, 'valores_distintos': int(len(np.unique(X_TODO[:, j]))),
                          'im_binaria': info_mutua(xd, YB_TODO), 'im_clase': info_mutua(xd, Y_TODO),
                          'pureza': pureza(xd, YB_TODO)})
        T1 = pd.DataFrame(filas)
        T1['im_binaria_rel'] = T1['im_binaria'] / H_yb
        T1['im_clase_rel'] = T1['im_clase'] / H_y
        T1['marcador'] = (T1['pureza'] >= UMBRAL_PUREZA) | (T1['im_binaria_rel'] >= UMBRAL_IM_REL)
        T1 = T1.sort_values('im_binaria', ascending=False).reset_index(drop=True)
        # Conflictos de etiqueta con todos los atributos
        h = pd.util.hash_pandas_object(pd.DataFrame(X_TODO), index=False).to_numpy()
        tab = pd.DataFrame({'h': h, 'y': Y_TODO})
        n_etq = tab.groupby('h')['y'].nunique()
        conflicto = tab['h'].map(n_etq).to_numpy() > 1
        cnt = tab.groupby(['h', 'y']).size().reset_index(name='n').sort_values(['h', 'n'], ascending=[True, False])
        cnt = cnt.drop_duplicates('h')
        mayoria = pd.Series(cnt['y'].to_numpy(), index=cnt['h'].to_numpy())
        pred_or = tab['h'].map(mayoria).to_numpy()
        ind_or = indicadores(Y_TODO, pred_or)
        A1 = {'H_binaria': H_yb, 'H_clase': H_y, 'pct_normal': float((Y_TODO == NORMAL).mean() * 100),
              'pureza_base': float(max((YB_TODO == 0).mean(), (YB_TODO == 1).mean())),
              'atributos': T1.to_dict(orient='records'), 'marcadores': T1.loc[T1['marcador'], 'atributo'].tolist(),
              'conflictos_pct': float(conflicto.mean() * 100), 'vectores_distintos': int(len(n_etq)),
              'oraculo': {k: ind_or[k] for k in ('f1_macro', 'tfp', 'ataques_como_normal_pct', 'exactitud')},
              'umbrales': {'pureza': UMBRAL_PUREZA, 'im_rel': UMBRAL_IM_REL}, 'segundos': round(time.time() - t0, 1)}
        guardar_json(A1, RUTA_A1)
    T1 = pd.DataFrame(A1['atributos'])
    print(f"H(Y_b) = {A1['H_binaria']:.4f} bits (Normal {A1['pct_normal']:.2f} %) · H(Y) = {A1['H_clase']:.4f} bits · "
          f"pureza de referencia (siempre la mayoritaria) = {A1['pureza_base']:.4f}")
    print('Marcadores candidatos:', A1['marcadores'] or 'ninguno')
    print(f"Conflictos de etiqueta: {A1['conflictos_pct']:.2f} % de las filas · oráculo: F1 macro {A1['oraculo']['f1_macro']:.2f} % · "
          f"TFP {A1['oraculo']['tfp']:.3f} %")
    mostrar(T1.head(15).round(4))

## A2. MI-LiteRF y MI-LiteXGB en 10 repeticiones
Por REP: división estratificada 70/30 (semilla 1000 + r), escalado min–max con el entrenamiento, ranking de IM en 200 000 filas de
entrenamiento. *k* se elige **una vez**, por CV en la REP-01 (XGBoost sin balanceo, regla de parsimonia) y se mantiene.
El balanceo *T* se elige por CV en cada REP y para cada modelo. La CV usa una submuestra estratificada de 400 000 filas de entrenamiento.

In [ ]:
def dividir(r):
    semilla = SEMILLA_BASE + r
    idx_tr, idx_te = train_test_split(np.arange(len(Y_TODO)), test_size=TEST_SIZE, stratify=Y_TODO, random_state=semilla)
    esc = MinMaxScaler().fit(X_TODO[idx_tr])
    return SimpleNamespace(r=r, codigo=f'REP-{r:02d}', semilla=semilla, idx_tr=idx_tr, idx_te=idx_te, esc=esc,
                           Xtr=esc.transform(X_TODO[idx_tr]).astype(np.float32), Xte=esc.transform(X_TODO[idx_te]).astype(np.float32),
                           ytr=Y_TODO[idx_tr], yte=Y_TODO[idx_te])

def submuestra(y, n, semilla):
    if len(y) <= n:
        return np.arange(len(y))
    idx, _ = train_test_split(np.arange(len(y)), train_size=n, stratify=y, random_state=semilla)
    return np.sort(idx)

def ranking_im(Dv, excluir=()):
    sub = submuestra(Dv.ytr, MI_SUBMUESTRA, Dv.semilla)
    Xs, ys = Dv.Xtr[sub], Dv.ytr[sub]
    im_ = []
    for j, a in enumerate(ATRIBUTOS):
        im_.append(-1.0 if a in excluir else info_mutua(discretizar(Xs[:, j]), ys))
    orden = np.argsort(-np.asarray(im_), kind='mergesort')
    orden = [j for j in orden if ATRIBUTOS[j] not in excluir]
    return orden, im_

def cv_puntaje(Dv, cols, fabrica, T, sub):
    skf = StratifiedKFold(3, shuffle=True, random_state=Dv.semilla)
    X, y = Dv.Xtr[sub][:, cols], Dv.ytr[sub]
    f1s, tam = [], []
    for k_, (a, b) in enumerate(skf.split(X, y)):
        Xb, yb = datos_T(X[a], y[a], T, Dv.semilla + k_)
        m = fabrica(Dv.semilla).fit(Xb, yb)
        f1s.append(f1_macro(y[b], m.predict(X[b]))); tam.append(kb_joblib(m))
    return float(np.mean(f1s)), float(np.mean(tam))

def regla_parsimonia(tabla, clave):
    mejor = tabla['f1_cv'].max()
    cand = tabla[tabla['f1_cv'] >= TOLERANCIA * mejor]
    return cand.sort_values(['tamano_kb', clave]).iloc[0]

RUTA_K = os.path.join(DIR_X, 'A2_eleccion_k.json')
def elegir_k():
    if hay(RUTA_K):
        return leer_json(RUTA_K)
    Dv = dividir(1); orden, _ = ranking_im(Dv); sub = submuestra(Dv.ytr, CV_SUBMUESTRA, Dv.semilla)
    filas = []
    for k in K_GRID:
        f1, tam = cv_puntaje(Dv, orden[:k], nuevo_xgb, None, sub)
        filas.append({'k': k, 'f1_cv': f1, 'tamano_kb': tam}); print(f'   k = {k}: F1 CV {f1:.2f} · {tam:.0f} KB')
    tabla = pd.DataFrame(filas)
    # parsimonia sobre k: el menor k con CV ≥ 99 % del mejor (como en el artículo)
    mejor = tabla['f1_cv'].max()
    k = int(tabla[tabla['f1_cv'] >= TOLERANCIA * mejor]['k'].min())
    res = {'k': k, 'tabla': filas, 'atributos_rep01': [ATRIBUTOS[j] for j in orden[:k]]}
    guardar_json(res, RUTA_K)
    return res

def a2_rep(r):
    dir_r = os.path.join(DIR_X, 'A2_arboles', f'REP-{r:02d}'); ruta = os.path.join(dir_r, 'resultado.json')
    if hay(ruta):
        return leer_json(ruta)
    os.makedirs(dir_r, exist_ok=True); t_ini = time.time()
    K = elegir_k()['k']
    Dv = dividir(r); orden, im_ = ranking_im(Dv); cols = orden[:K]
    sub = submuestra(Dv.ytr, CV_SUBMUESTRA, Dv.semilla)
    res = {'rep': r, 'semilla': Dv.semilla, 'k': K, 'atributos_k': [ATRIBUTOS[j] for j in cols],
           'ranking': [{'atributo': ATRIBUTOS[j], 'im': float(im_[j])} for j in orden], 'n_train': int(len(Dv.ytr)),
           'n_test': int(len(Dv.yte)), 'modelos': {}}
    for nombre in ('rf', 'xgb'):
        filas = []
        for T in T_GRID:
            f1, tam = cv_puntaje(Dv, cols, FABRICA[nombre], T, sub)
            filas.append({'T': T if T else 0, 'f1_cv': f1, 'tamano_kb': tam})
        tabla = pd.DataFrame(filas)
        T = int(regla_parsimonia(tabla, 'T')['T']) or None
        Xb, yb = datos_T(Dv.Xtr[:, cols], Dv.ytr, T, Dv.semilla)
        t0 = time.perf_counter(); m = FABRICA[nombre](Dv.semilla).fit(Xb, yb); t_ent = time.perf_counter() - t0
        pred = m.predict(Dv.Xte[:, cols])
        res['modelos'][nombre] = {'T': T or 0, 'cv': filas, 'tiempo_entrenamiento_s': t_ent, 'tamano_kb': kb_joblib(m),
                                  **indicadores(Dv.yte, pred)}
        if r == 1:
            joblib.dump(m, os.path.join(dir_r, f'{nombre}.joblib'))
        print(f"   {nombre.upper()}: T = {T_txt(T)} · F1 macro {res['modelos'][nombre]['f1_macro']:.2f} % · "
              f"TFP {res['modelos'][nombre]['tfp']:.3f} % · {t_ent:.0f} s")
    if r == 1:
        np.savez(os.path.join(dir_r, 'escalador.npz'), min_=Dv.esc.min_, scale_=Dv.esc.scale_, cols=np.asarray(cols))
    res['segundos'] = round(time.time() - t_ini, 1)
    guardar_json(res, ruta)
    return res

if correr('A2'):
    print('Elección de k (REP-01, CV):'); EK = elegir_k(); print(f"   k = {EK['k']}")
    A2 = {}
    for r in REPS:
        print(f'REP-{r:02d}'); A2[r] = a2_rep(r)

## A3. TCN de Gao et al. (GPU), sin cambios, con las mismas entradas de IM
Misma arquitectura y configuración del artículo (4 bloques residuales, dilataciones 1–8, 26 filtros, núcleo 1, Adam 10⁻³,
50 épocas, lote 128, 10 % de validación, sin balanceo). Se entrena en un proceso aparte que también predice la prueba.

In [ ]:
RUTA_TCN = os.path.join(DIR_TMP, 'entrenar_tcn_s14.py')
SCRIPT_TCN = r'''
import argparse, json, time
import numpy as np
p = argparse.ArgumentParser()
for a_ in ('--x', '--y', '--xte', '--pred', '--modelo', '--info', '--cfg'):
    p.add_argument(a_)
p.add_argument('--semilla', type=int)
p.add_argument('--n_cls', type=int)
a = p.parse_args()
import tensorflow as tf
import keras
from keras import layers
cfg = json.loads(a.cfg)

def construir_tcn(n_atr, n_cls, filtros, kernel, dilataciones, dropout):   # la del artículo, sin cambios
    ent = keras.Input(shape=(1, n_atr))
    x = ent
    for d in dilataciones:
        h = layers.Conv1D(filtros, kernel, dilation_rate=d, padding='causal')(x)
        h = layers.BatchNormalization()(h)
        h = layers.PReLU(shared_axes=[1])(h)
        h = layers.Dropout(dropout)(h)
        res = x if x.shape[-1] == filtros else layers.Conv1D(filtros, 1)(x)
        x = layers.PReLU(shared_axes=[1])(layers.Add()([res, h]))
    x = layers.Flatten()(x)
    sal = layers.Dense(n_cls, activation='softmax')(x)
    return keras.Model(ent, sal, name='TCN_Gao2026')

keras.utils.set_random_seed(a.semilla)
X, y = np.load(a.x), np.load(a.y)
tcn = construir_tcn(X.shape[1], a.n_cls, cfg['filtros'], cfg['kernel'], cfg['dilataciones'], cfg['dropout'])
tcn.compile(optimizer=keras.optimizers.Adam(cfg['lr']), loss='categorical_crossentropy', metrics=['accuracy'])
t0 = time.perf_counter()
hist = tcn.fit(X.reshape(-1, 1, X.shape[1]), keras.utils.to_categorical(y, a.n_cls), epochs=cfg['epocas'], batch_size=cfg['lote'],
               validation_split=cfg['val_split'], shuffle=True, verbose=2)
t_ent = time.perf_counter() - t0
tcn.save(a.modelo)
Xte = np.load(a.xte)
pred = np.argmax(tcn.predict(Xte.reshape(-1, 1, Xte.shape[1]), batch_size=8192, verbose=0), axis=1).astype(np.int16)
np.save(a.pred, pred)
json.dump({'tiempo_entrenamiento_s': round(t_ent, 1), 'parametros': int(tcn.count_params()), 'config': cfg,
           'epocas_corridas': len(hist.history['loss']), 'gpu': [g.name for g in tf.config.list_physical_devices('GPU')]},
          open(a.info, 'w'), indent=2)
print('TCN lista:', round(t_ent / 60, 1), 'min')
'''
with open(RUTA_TCN, 'w') as f:
    f.write(SCRIPT_TCN)

def a3_rep(r):
    dir_r = os.path.join(DIR_X, 'A3_tcn', f'REP-{r:02d}'); ruta = os.path.join(dir_r, 'resultado.json')
    if hay(ruta):
        return leer_json(ruta)
    ruta_a2 = os.path.join(DIR_X, 'A2_arboles', f'REP-{r:02d}', 'resultado.json')
    if not os.path.exists(ruta_a2):
        raise SystemExit(f'⛔ Falta A2 de la REP-{r:02d} (fija las entradas de IM). Ejecuta A2 primero.')
    t_ini = time.time(); os.makedirs(dir_r, exist_ok=True)
    cols = [ATRIBUTOS.index(a) for a in leer_json(ruta_a2)['atributos_k']]
    Dv = dividir(r)
    tmp = os.path.join(DIR_TMP, 'A3', f'REP-{r:02d}'); os.makedirs(tmp, exist_ok=True)
    x, y, xte, pred = (os.path.join(tmp, n) for n in ('xtr.npy', 'ytr.npy', 'xte.npy', 'pred.npy'))
    np.save(x, np.ascontiguousarray(Dv.Xtr[:, cols])); np.save(y, Dv.ytr); np.save(xte, np.ascontiguousarray(Dv.Xte[:, cols]))
    info = os.path.join(dir_r, 'entrenamiento.json'); log = os.path.join(tmp, 'log.txt')
    modelo = os.path.join(dir_r if r == 1 else tmp, 'tcn.keras')
    cmd = [sys.executable, RUTA_TCN, '--x', x, '--y', y, '--xte', xte, '--pred', pred, '--modelo', modelo, '--info', info,
           '--cfg', json.dumps(TCN_CFG), '--semilla', str(Dv.semilla), '--n_cls', str(N_CLASES)]
    env = dict(os.environ, TF_FORCE_GPU_ALLOW_GROWTH='true', TF_CPP_MIN_LOG_LEVEL='2')
    print(f'   entrenando la TCN (avance en {log})')
    p = subprocess.run(cmd, stdout=open(log, 'w'), stderr=subprocess.STDOUT, env=env)
    if p.returncode != 0 or not os.path.exists(pred):
        print('\n'.join(open(log).read().splitlines()[-40:]))
        raise RuntimeError(f'Falló la TCN en la REP-{r:02d}')
    res = {'rep': r, 'k': len(cols), 'atributos': [ATRIBUTOS[j] for j in cols], **indicadores(Dv.yte, np.load(pred).astype(np.int64)),
           'entrenamiento': leer_json(info), 'segundos': round(time.time() - t_ini, 1)}
    guardar_json(res, ruta); shutil.copy(log, dir_r); shutil.rmtree(tmp, ignore_errors=True)
    return res

if correr('A3'):
    pendientes = [r for r in REPS if not hay(os.path.join(DIR_X, 'A3_tcn', f'REP-{r:02d}', 'resultado.json'))]
    if pendientes and not hay_gpu() and not MODO_PRUEBA:
        raise SystemExit('⛔ A3 necesita GPU (Entorno de ejecución → Cambiar tipo de entorno → L4 o A100, RAM alta).')
    A3 = {}
    for r in REPS:
        t0 = time.time(); A3[r] = a3_rep(r)
        print(f"REP-{r:02d} TCN: F1 macro {A3[r]['f1_macro']:.2f} % · TFP {A3[r]['tfp']:.3f} % · {time.time() - t0:.0f} s")

## A4. Ablación de marcadores (XGBoost sin balanceo)
Se reentrena MI-LiteXGB (mismo *k*, sin balanceo) quitando del ranking los marcadores candidatos de A1; si A1 no señala
ninguno, se quita el atributo de mayor IM de cada REP, como prueba de sensibilidad. Diferencias pareadas con la t remuestreada corregida.

In [ ]:
def a4_rep(r, variante, excluir):
    dir_r = os.path.join(DIR_X, 'A4_ablacion', variante); ruta = os.path.join(dir_r, f'REP-{r:02d}.json')
    if hay(ruta):
        return leer_json(ruta)
    K = elegir_k()['k']; Dv = dividir(r)
    if variante == 'sin_top1':
        orden0, _ = ranking_im(Dv); excluir = [ATRIBUTOS[orden0[0]]]
    orden, _ = ranking_im(Dv, excluir=set(excluir)); cols = orden[:K]
    m = nuevo_xgb(Dv.semilla).fit(Dv.Xtr[:, cols], Dv.ytr)
    res = {'rep': r, 'variante': variante, 'excluidos': list(excluir), 'atributos_k': [ATRIBUTOS[j] for j in cols],
           **indicadores(Dv.yte, m.predict(Dv.Xte[:, cols]))}
    guardar_json(res, ruta)
    return res

if correr('A4'):
    A1 = leer_json(RUTA_A1)
    # Si A1 señala marcadores, se quitan esos; si no señala ninguno, se quita el atributo de mayor IM (sensibilidad)
    variantes = [('sin_marcadores', A1['marcadores'])] if A1['marcadores'] else [('sin_top1', [])]
    A4 = {}
    for v, ex in variantes:
        for r in REPS:
            A4[(v, r)] = a4_rep(r, v, ex)
            print(f"REP-{r:02d} {v} (sin {', '.join(A4[(v, r)]['excluidos'])}): F1 macro {A4[(v, r)]['f1_macro']:.2f} %")

## A5. Latencia por paquete (REP-01, ONNX Runtime, 1 hilo, lote de 1)
Mediana sobre 10 000 paquetes de prueba tras 500 llamadas de calentamiento, cada modelo en su propio proceso, como en el artículo.

In [ ]:
RUTA_BENCH = os.path.join(DIR_TMP, 'bench_s14.py')
with open(RUTA_BENCH, 'w') as f:
    f.write(r'''
import sys, json, time, numpy as np, onnxruntime as ort, resource
modelo, xpath, salida, n, cal = sys.argv[1], sys.argv[2], sys.argv[3], int(sys.argv[4]), int(sys.argv[5])
so = ort.SessionOptions(); so.intra_op_num_threads = 1; so.inter_op_num_threads = 1
so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
s = ort.InferenceSession(modelo, so, providers=['CPUExecutionProvider'])
ent = s.get_inputs()[0]; es3 = len(ent.shape) == 3
X = np.load(xpath).astype(np.float32)[:n]
def pr(x):
    return s.run(None, {ent.name: x.reshape(len(x), 1, -1) if es3 else x})[0]
for i in range(min(cal, len(X))):
    pr(X[i:i + 1])
t = np.empty(len(X)); out = []
for i in range(len(X)):
    c = time.perf_counter_ns(); o = pr(X[i:i + 1]); t[i] = time.perf_counter_ns() - c; out.append(o)
o = np.concatenate(out)
pred = np.argmax(o, axis=1) if o.ndim == 2 else o.astype(np.int64).ravel()
json.dump({'mediana_us': float(np.median(t) / 1e3), 'p99_us': float(np.percentile(t, 99) / 1e3), 'media_us': float(t.mean() / 1e3),
           'pico_mb': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 'pred': pred.tolist()}, open(salida, 'w'))
''')

def a_onnx(nombre, modelo_o_ruta, k, destino):
    if nombre == 'rf':
        from skl2onnx import convert_sklearn
        from skl2onnx.common.data_types import FloatTensorType
        on = convert_sklearn(modelo_o_ruta, initial_types=[('x', FloatTensorType([None, k]))],
                             options={id(modelo_o_ruta): {'zipmap': False}}, target_opset={'': 17, 'ai.onnx.ml': 3})
        open(destino, 'wb').write(on.SerializeToString())
    elif nombre == 'xgb':
        import onnxmltools
        from onnxmltools.convert.common.data_types import FloatTensorType
        on = onnxmltools.convert_xgboost(modelo_o_ruta, initial_types=[('x', FloatTensorType([None, k]))], target_opset=15)
        open(destino, 'wb').write(on.SerializeToString())
    else:
        import tensorflow as tf, keras, tf2onnx
        m = keras.models.load_model(modelo_o_ruta)
        try:
            tf2onnx.convert.from_keras(m, input_signature=(tf.TensorSpec((None, 1, k), tf.float32, name='x'),), opset=17,
                                       output_path=destino)
        except Exception:
            m.export(destino, format='onnx')
    return destino

RUTA_A5 = os.path.join(DIR_X, 'A5_latencia.json')
if correr('A5'):
    if hay(RUTA_A5):
        A5 = leer_json(RUTA_A5)
    else:
        d1 = os.path.join(DIR_X, 'A2_arboles', 'REP-01'); dt = os.path.join(DIR_X, 'A3_tcn', 'REP-01')
        Dv = dividir(1); cols = [ATRIBUTOS.index(a) for a in leer_json(os.path.join(d1, 'resultado.json'))['atributos_k']]
        sub = np.random.default_rng(1).choice(len(Dv.yte), size=min(N_LOTE1, len(Dv.yte)), replace=False)
        xpath = os.path.join(DIR_TMP, 'x_lote1.npy'); np.save(xpath, Dv.Xte[sub][:, cols])
        A5 = {'procesador': cpu_modelo(), 'n': int(len(sub)), 'modelos': {}}
        for nombre, fuente in (('rf', os.path.join(d1, 'rf.joblib')), ('xgb', os.path.join(d1, 'xgb.joblib')), ('tcn', os.path.join(dt, 'tcn.keras'))):
            if not os.path.exists(fuente):
                print(f'   {nombre}: falta {fuente}, se omite'); continue
            try:
                destino = os.path.join(DIR_TMP, f'{nombre}.onnx')
                a_onnx(nombre, joblib.load(fuente) if nombre != 'tcn' else fuente, len(cols), destino)
                sal = os.path.join(DIR_TMP, f'lat_{nombre}.json')
                env = dict(os.environ, OMP_NUM_THREADS='1', CUDA_VISIBLE_DEVICES='-1')
                subprocess.run([sys.executable, RUTA_BENCH, destino, xpath, sal, str(N_LOTE1), str(CALENTAMIENTO)], check=True, env=env)
                L = leer_json(sal)
                if nombre != 'tcn':
                    nativo = joblib.load(fuente).predict(Dv.Xte[sub][:, cols])
                    L['acuerdo_nativo'] = float((np.asarray(L['pred']) == np.asarray(nativo)).mean())
                L.pop('pred'); L['onnx_kb'] = os.path.getsize(destino) / 1024
                A5['modelos'][nombre] = L
                print(f"   {nombre.upper()}: mediana {L['mediana_us']:.1f} µs · p99 {L['p99_us']:.1f} µs · ONNX {L['onnx_kb']:.0f} KB")
            except Exception as e:
                print(f'   {nombre}: no se pudo medir ({type(e).__name__}: {str(e)[:150]})')
        guardar_json(A5, RUTA_A5)

## A6. Resumen y pruebas pareadas
Media ± DE en las REP disponibles, pruebas con la t remuestreada corregida (Holm) y el archivo `S14_CICIoT2023_resultados.json`.
**Comparte este archivo** para integrar los resultados al artículo.

In [ ]:
if correr('A6'):
    A0 = leer_json(RUTA_A0); A1 = leer_json(RUTA_A1) if os.path.exists(RUTA_A1) else None
    leer = lambda parte, r: (leer_json(p) if os.path.exists(p := os.path.join(DIR_X, parte, f'REP-{r:02d}', 'resultado.json')) else None)
    R2 = {r: leer('A2_arboles', r) for r in range(1, 11)}; R3 = {r: leer('A3_tcn', r) for r in range(1, 11)}
    reps_ok = [r for r in range(1, 11) if R2[r] and R3[r]]
    print('REP completas (árboles y TCN):', reps_ok)
    filas = []
    for r in range(1, 11):
        for nombre, d in (('MI-LiteRF', R2[r] and R2[r]['modelos']['rf']), ('MI-LiteXGB', R2[r] and R2[r]['modelos']['xgb']), ('TCN', R3[r])):
            if d:
                filas.append({'rep': r, 'modelo': nombre, **{k: d[k] for k in ('f1_macro', 'exactitud', 'f1_ponderado', 'tfp',
                                                                                 'falsas_alarmas', 'ataques_como_normal_pct', 'E_N_registros')},
                              'T': d.get('T', 0)})
    Tab = pd.DataFrame(filas)
    resumen = {m: {k: ms(g[k].dropna()) for k in ('f1_macro', 'exactitud', 'f1_ponderado', 'tfp', 'ataques_como_normal_pct', 'E_N_registros')}
               for m, g in Tab.groupby('modelo')}
    for m in resumen:
        g = Tab[Tab['modelo'] == m]
        resumen[m]['n_rep'] = int(len(g)); resumen[m]['T_elegidos'] = g['T'].map(T_txt).value_counts().to_dict()
    n_tr = R2[reps_ok[0]]['n_train'] if reps_ok else 1; n_te = R2[reps_ok[0]]['n_test'] if reps_ok else 1
    pruebas = []
    if len(reps_ok) >= 3:
        par = lambda a, b, k: np.array([b(r)[k] - a(r)[k] for r in reps_ok])
        xg = lambda r: R2[r]['modelos']['xgb']; rf = lambda r: R2[r]['modelos']['rf']; tc = lambda r: R3[r]
        for ind in ('f1_macro', 'tfp', 'ataques_como_normal_pct'):
            for nom, a, b in (('TCN → MI-LiteXGB', tc, xg), ('TCN → MI-LiteRF', tc, rf), ('MI-LiteRF → MI-LiteXGB', rf, xg)):
                pruebas.append({'indicador': ind, 'comparacion': nom, **t_corregida(par(a, b, ind), n_tr, n_te)})
        p_h = holm([x['p_corr'] for x in pruebas])
        for x, ph in zip(pruebas, p_h):
            x['p_holm'] = float(ph)
    ablacion = {}
    for v in ('sin_marcadores', 'sin_top1'):
        rs = [r for r in reps_ok if os.path.exists(os.path.join(DIR_X, 'A4_ablacion', v, f'REP-{r:02d}.json'))]
        if len(rs) >= 2:
            A4v = {r: leer_json(os.path.join(DIR_X, 'A4_ablacion', v, f'REP-{r:02d}.json')) for r in rs}
            d = np.array([A4v[r]['f1_macro'] - R2[r]['modelos']['xgb']['f1_macro'] for r in rs])
            ablacion[v] = {'excluidos': sorted({a for r in rs for a in A4v[r]['excluidos']}), 'f1_macro': ms([A4v[r]['f1_macro'] for r in rs]),
                           'tfp': ms([A4v[r]['tfp'] for r in rs]), 'delta_vs_xgb': t_corregida(d, n_tr, n_te), 'reps': rs}
    # F1 por clase y matrices sumadas
    pc = {}
    for nombre, fuente in (('MI-LiteRF', lambda r: R2[r]['modelos']['rf']), ('MI-LiteXGB', lambda r: R2[r]['modelos']['xgb']), ('TCN', lambda r: R3[r])):
        if reps_ok:
            pc[nombre] = {'f1_por_clase': {c: float(np.mean([fuente(r)['f1_por_clase'][c] for r in reps_ok])) for c in CLASES},
                          'matriz_sumada': np.sum([np.array(fuente(r)['matriz_confusion']) for r in reps_ok], axis=0).tolist()}
    A5 = leer_json(RUTA_A5) if os.path.exists(RUTA_A5) else None
    K = leer_json(RUTA_K) if os.path.exists(RUTA_K) else None
    S = {'generado': datetime.datetime.now().strftime('%d/%m/%Y %H:%M'), 'entorno': ENTORNO, 'tarea': TAREA, 'clases': CLASES,
         'datos': {k: A0[k] for k in A0 if k not in ('atributos',)}, 'n_atributos': len(ATRIBUTOS),
         'configuracion': {'reps': reps_ok, 'test_size': TEST_SIZE, 'mi_submuestra': MI_SUBMUESTRA, 'cv_submuestra': CV_SUBMUESTRA,
                           'k_grid': K_GRID, 'T_grid': [t or 0 for t in T_GRID], 'rf': RF_CFG, 'xgb': XGB_CFG, 'tcn': TCN_CFG},
         'eleccion_k': K, 'auditoria': A1, 'resumen': resumen, 'pruebas': pruebas, 'ablacion': ablacion, 'por_clase': pc,
         'latencia': A5, 'por_rep': Tab.to_dict(orient='records')}
    ruta = os.path.join(DIR_X, 'S14_CICIoT2023_resultados.json'); guardar_json(S, ruta)
    print(f'\n=== CICIoT2023 ({TAREA}, {A0["archivos_usados"]} archivos, {A0["filas_depuradas"]:,} filas, k = {K and K["k"]}) ===')
    for m, v in resumen.items():
        print(f"{m:11s} F1 macro {v['f1_macro'][0]:.2f} ± {v['f1_macro'][1]:.2f} · exactitud {v['exactitud'][0]:.2f} · "
              f"TFP {v['tfp'][0]:.3f} ± {v['tfp'][1]:.3f} · ataques como Normal {v['ataques_como_normal_pct'][0]:.2f} % · n = {v['n_rep']}")
    for x in pruebas:
        if x['indicador'] == 'f1_macro':
            print(f"  {x['comparacion']}: Δ F1 {x['media_d']:+.2f} {x['ic95_corr']} · p_corr (Holm) {x['p_holm']:.2g}")
    for v, a in ablacion.items():
        print(f"  Ablación {v} (sin {', '.join(a['excluidos'])}): F1 {a['f1_macro'][0]:.2f} · Δ {a['delta_vs_xgb']['media_d']:+.2f} "
              f"{a['delta_vs_xgb']['ic95_corr']}")
    if A1:
        print('  Marcadores de la auditoría:', A1['marcadores'] or 'ninguno', f"· conflictos {A1['conflictos_pct']:.2f} %")
    print('\n✔ Guardado:', ruta)